# Command Over Serial – Minino BLE

The **Minino BLE firmware** provides a serial interface for exploring Bluetooth Low Energy (BLE) devices through GATT operations such as discovery, reading, and writing.  

In this section of the notebook, you will:  
1. Identify and select the USB serial port where your Minino is connected.  
2. Establish a serial connection at the correct baud rate.  
3. Send commands to the device and observe its responses.  

As a starting point, we’ll demonstrate the use of the command:  

gatt_cmdscan

This command instructs Minino to begin scanning for nearby BLE devices and report their advertising information. Once confirmed, you can build on this workflow to issue additional GATT-related commands for deeper exploration.


In [1]:
# Automatically detect your USB serial port
import serial.tools.list_ports

def detect_serial_port():
    ports = serial.tools.list_ports.comports()
    for port in ports:
        if any(x in port.device for x in ["ttyUSB", "ttyACM", "cu.usbmodem", "usbserial", "COM"]):
            return port.device
    return None

selected_port = detect_serial_port()
if selected_port:
    print(f"Serial Port: {selected_port}")
else:
    print("No serial port detected.")
    

Serial Port: /dev/ttyACM0


In [2]:
# Detect the Minino Board
import serial, time

serial_port = selected_port if 'selected_port' in globals() else '/dev/cu.usbmodem2101'

serial_conn = None

is_a_minino_board = False

# Function to connect the Serial
def connect_serial():
    global serial_conn
    # Connect with the port
    try:
        serial_conn = serial.Serial(serial_port, 115200, timeout=1)
        time.sleep(2)
        return True
    except Exception as e:
        return False

# Function to send a command via Serial with a response 
def send_command_string_with_response(command:str):
    global serial_conn

    # Send any word to get the return
    if serial_conn and serial_conn.is_open:
        cmd = command+'\r\n'
        serial_conn.write(cmd.encode())
        time.sleep(0.3)
        try: 
            resp = serial_conn.read_all().decode(errors='ignore')
            return resp
        except:
            return 'Nan'
            
    else:
        return 'Nan'

# Function to disconnect Serial
def disconnect_serial():
    global serial_conn
    
    # Disconnect with the port
    if serial_conn and serial_conn.is_open:
        serial_conn.close()
        return True
    else: 
        return False

# Function to detect the minino
def detect_minino():
    connect_serial()
    data = send_command_string_with_response('h')
    if "minino" in data:
        is_a_minino_board = True
        print(f"Minino Detected at {serial_port}")
    else:
        print("Minino no detected")
    disconnect_serial()

# Test for minino
detect_minino()

Minino Detected at /dev/ttyACM0


In [3]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import threading

# Widgets Added on the UI
output_serial = widgets.Output()
command_input = widgets.Text(description='Comando:')
send_button = widgets.Button(description='Enviar')
close_button = widgets.Button(description='Cerrar Puerto', button_style='danger')
open_button = widgets.Button(description='Abrir Puerto', button_style='success')
auto_button = widgets.Button(description='▶️ Auto GATT Scan', button_style='info')

# Open Serial with button open
def open_serial(b=None):
    if (connect_serial()):  
        with output_serial:
            clear_output()
            print(f"Connect at {serial_port}")
    else:
        with output_serial:
            clear_output()
            print("Not connected")

# Close serial with button close
def close_serial(b=None):
    if (disconnect_serial()):
        with output_serial:
            clear_output()
            print(f"Device disconnect at {serial_port}")
    else:
        with output_serial:
            clear_output()
            print("Not device")

# send a command from the label
def send_command_label(b):
    cmd = command_input.value.strip()
    response = send_command_string_with_response(cmd)
    with output_serial:
            clear_output()
            print(f"{response}")

# Send the auto_gatt_scan
def auto_gatt_scan(b=None):
    global serial_conn

    # Check if the serial is open
    if not serial_conn or not serial_conn.is_open:
        if not connect_serial():
            with output_serial:
                clear_output()
                print("Device not connected")
            return

    # Display a wait time 
    with output_serial:
        clear_output()
        print("Wait for the response...")

    # Send the command to start the gatt scan
    serial_conn.write(b'gattcmd_scan\r\n')

    # Take the last time 
    last_time = time.monotonic()

    # Buffer to save the response from the Minino
    buffer = ""

    # Get the responses
    while (time.monotonic() - last_time) < 5:
        try:
            response = serial_conn.read_all().decode(errors='ignore')
            if response:
                buffer+=response
        except:
            with output_serial:
                clear_output()
                print("Error")
            return

    # Stop the scan on Minino
    serial_conn.write(b'\x03') # By the moment is not working, the Minino is not taking the ctrl+c command 

    # Display buffer
    with output_serial:
        clear_output()
        print(buffer)

open_button.on_click(open_serial)
close_button.on_click(close_serial)
send_button.on_click(send_command_label)
auto_button.on_click(auto_gatt_scan)

# Display UI
display(widgets.HBox([open_button, close_button, auto_button]))
display(command_input, send_button, output_serial)


Text(value='', description='Comando:')

Button(description='Enviar', style=ButtonStyle())

Output()